# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced using their `@id` in accordance with Croissant best practices.

### Dataset Source
The dataset schema is available at:

[https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the Croissant dataset schema and its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset's Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (do not subscript the metadata object)
print(f"Dataset loaded: {dataset.metadata.name}")
print(f"\nDescription: {dataset.metadata.description}")
print(f"\nVersion: {dataset.metadata.version}")


## 2. Data Overview
Review available record sets, their IDs, and the fields within each. All references use `@id` to ensure unambiguous selection.

In [ ]:
# Enumerate record sets (tables) and their fields by @id
record_sets = dataset.metadata.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    if 'field' in rs:
        print("  Fields (by @id):")
        # Fields may be a list of dicts (objects)
        for field in rs['field']:
            print(f"    {field['@id']}")
    print("")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# --- Choose record set IDs ---
# For this dataset, the record set IDs usually follow the pattern 'cr:RecordSet/<name>'.
# Let's automatically obtain the @id for all available record sets.
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Sample:\n{df.head(2)}\n")
    else:
        print("  No records found for this set.\n")


## 4. Exploratory Data Analysis (EDA)
Process a chosen data table with typical EDA tasks: filtering by a numeric field, normalizing, and grouping. All fields referenced by `@id`.


In [ ]:
# --- Example EDA on the main protein record set ---
# Select the main protein record set (manually, usually the biggest one)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"Using record set: {main_record_set_id}\nColumns: {df.columns.tolist()}")
    # Choose a numeric field by @id (e.g., coverage_percentage or molecular_weight_mw)
    # We'll select the first numeric-type column detected (float/int)
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        raise ValueError("No numeric columns found for EDA.")

    print(f"\nAnalyzing numeric field: {numeric_field}")

    # Filter out non-NaN and apply a threshold (for demonstration, use 10th percentile)
    threshold = df[numeric_field].quantile(0.1)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows\n")

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
        filtered_df[numeric_field].std()
    )
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a likely categorical field (the first object/string column)
    group_field = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped by {group_field}, mean values:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field available for grouping.")
else:
    print("No usable record sets detected.")

## 5. Visualization
Visualize the distribution of the main numeric field and, if possible, its normalized version across groups.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution
if record_set_ids and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True, bins=30, color='orange')
        plt.title(f"Normalized Distribution of {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Frequency")
        plt.show()

    # If there is a group field, show boxplot
    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(
            y=filtered_df[group_field],
            x=filtered_df[numeric_field],
            orient='h',
        )
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(numeric_field)
        plt.ylabel(group_field)
        plt.show()


## 6. Conclusion
In this notebook, we loaded the Mass Spectrometry Extracellular Vesicle dataset using the `mlcroissant` library, reviewed record sets and field `@id` values, extracted tabular data, performed basic EDA with normalization and grouping, and visualized key data distributions.

Future analysis can focus on specific proteins, peptide identification, or downstream machine learning with these biologically relevant measurements.